# Predicting arrears next month

**Goal:** predict `arrears_next_month` from information known at the end of a billing month.

Design decisions (following `checklist.md`):
- The model is meant for future months, and accounts (`meter_ref`) appear in many rows. So the test set is the **latest months, for accounts held out entirely** (no test account appears in training at any date).
- `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time. It is dropped (leakage). `meter_ref` is an identifier and is dropped. `billing_month` is not used as a feature either, as the arrears rate drifts strongly over time and a date cannot be extrapolated.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
df=pd.read_csv('data.csv',parse_dates=['billing_month'])
print(df.shape, df.meter_ref.nunique(), df.isna().sum().sum())
df.groupby('billing_month').arrears_next_month.mean().plot(title='Arrears rate by month (strong drift)');

## 1. Test set (set aside before modelling)
Latest 4 months (Sep–Dec 2025) AND a random 20% of accounts. Training = earlier months, other accounts only. Rows of training accounts after the cutoff and of test accounts before it are discarded, so the two sets share no accounts and no period.

In [ ]:
rng=np.random.RandomState(0)
accts=df.meter_ref.unique()
test_accts=set(rng.choice(accts,int(0.2*len(accts)),replace=False))
months=np.sort(df.billing_month.unique())
cutoff=months[-4]
is_test_acct=df.meter_ref.isin(test_accts)
train=df[~is_test_acct & (df.billing_month<cutoff)].reset_index(drop=True)
test=df[is_test_acct & (df.billing_month>=cutoff)].reset_index(drop=True)
print(len(train),len(test),train.meter_ref.nunique(),test.meter_ref.nunique())
print('base rate train %.3f test %.3f'%(train.arrears_next_month.mean(),test.arrears_next_month.mean()))
assert not set(train.meter_ref)&set(test.meter_ref)
# test is not touched again until the final step

## 2. Features and candidate models (named before any scoring)
Features: `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.

Candidates:
1. **baseline** – predicts the training prior for everyone (no skill, PR-AUC = base rate)
2. **logreg** – one-hot tariff, scaled numerics, L2 logistic regression (C=1)
3. **hgb** – HistGradientBoosting (lr 0.05, max_depth 3, 200 iter)
4. **logreg_late_only** – logistic regression on `late_payments_12m` alone

Selection metric: mean cross-validated PR-AUC (positive class is a minority and drifts). Decision threshold: the one maximising F1 on pooled out-of-fold predictions.

In [ ]:
FEATS=['tariff','occupants','kwh','bill_eur','late_payments_12m']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
ohe=lambda: OneHotEncoder(handle_unknown='ignore')
models={
 'baseline':(DummyClassifier(strategy='prior'),FEATS),
 'logreg':(make_pipeline(ColumnTransformer([('c',ohe(),['tariff']),('n',StandardScaler(),NUM)]),LogisticRegression(C=1,max_iter=1000)),FEATS),
 'hgb':(make_pipeline(ColumnTransformer([('c',ohe(),['tariff'])],remainder='passthrough'),
        HistGradientBoostingClassifier(learning_rate=0.05,max_depth=3,max_iter=200,random_state=0)),FEATS),
 'logreg_late_only':(make_pipeline(ColumnTransformer([('n',StandardScaler(),['late_payments_12m'])]),LogisticRegression(max_iter=1000)),['late_payments_12m']),
}

## 3. Cross-validation inside training rows
Forward-chaining over time with whole accounts held out: for each fold, validation rows are one block of later months from a random half of accounts; training rows are all earlier months from the *other* half. Scaling/encoding live inside the pipelines, so they are fit per fold.

In [ ]:
tm=np.sort(train.billing_month.unique())
blocks=np.array_split(tm[len(tm)//2:],5)   # 5 validation blocks in the later half of the training period
tacc=train.meter_ref.unique()
def folds():
    for k,b in enumerate(blocks):
        r=np.random.RandomState(100+k); val_acc=set(r.choice(tacc,len(tacc)//2,replace=False))
        va=train.meter_ref.isin(val_acc)
        yield k,train.index[~va & (train.billing_month<b[0])],train.index[va & train.billing_month.isin(b)]
oof={n:[] for n in models}; ys=[]; rows=[]
y=train.arrears_next_month
for k,tr,va in folds():
    ys.append(y[va].values)
    for n,(m,f) in models.items():
        p=clone(m).fit(train.loc[tr,f],y[tr]).predict_proba(train.loc[va,f])[:,1]
        oof[n].append(p)
        rows.append((n,k,average_precision_score(y[va],p),roc_auc_score(y[va],p),y[va].mean()))
cv=pd.DataFrame(rows,columns=['model','fold','pr_auc','roc_auc','fold_base_rate'])
print(cv.pivot(index='fold',columns='model',values='pr_auc').round(3))
print(cv.groupby('model')[['pr_auc','roc_auc']].agg(['mean','std']).round(3))
print('fold base rates',cv[cv.model=='baseline'].fold_base_rate.round(3).tolist())

## 4. Choice (made on CV only)
The non-baseline model with the highest mean CV PR-AUC is chosen; the threshold maximises F1 on pooled out-of-fold predictions of that model.

In [ ]:
best=cv.groupby('model').pr_auc.mean().drop('baseline').idxmax()
yo=np.concatenate(ys); po=np.concatenate(oof[best])
ths=np.linspace(0.02,0.9,89); f1s=[f1_score(yo,po>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('CHOSEN MODEL:',best,'| threshold:',round(thr,2),'| CV F1 at threshold: %.3f'%max(f1s))

**Recorded choice:** the model and threshold printed above are final. No test data has been scored. The next cell refits on all training rows and scores the test set once.

In [ ]:
m,f=models[best]
final=clone(m).fit(train[f],train.arrears_next_month)
yt=test.arrears_next_month.values
pt=final.predict_proba(test[f])[:,1]
pred=pt>=thr
res=pd.Series({'test base rate (= baseline PR-AUC)':yt.mean(),'PR-AUC (model)':average_precision_score(yt,pt),
 'ROC-AUC':roc_auc_score(yt,pt),'precision@thr':precision_score(yt,pred,zero_division=0),'recall@thr':recall_score(yt,pred),'F1@thr':f1_score(yt,pred)})
print(res.round(3))

## Conclusion
See the output above for the headline test numbers: PR-AUC against the test base rate (what a no-skill model scores). Caveats: the arrears rate rises steeply through 2025, so the test period's base rate is much higher than in most training months, which hurts threshold calibration (precision/recall at the CV threshold) more than ranking (AUC). The test set has few accounts, so numbers are noisy. `final_notice_sent` would probably be very predictive but is not known at prediction time, so it is excluded.